In [12]:
import torch, gc
def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

free_gpu()

In [13]:
# !pip install sentence_transformers
# !pip install rdflib

In [14]:
# from google.colab import drive
# drive.mount('/content/drive')

In [15]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
import pandas as pd


class ModelIterator:
    def __init__(self, models_name):
        self.models_name = models_name
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        free_gpu()
        if self.index >= len(self.models_name):
            raise StopIteration
        model = SentenceTransformer(self.models_name[self.index])
        self.index += 1
        return model

    def __getitem__(self, index):
        free_gpu()
        return SentenceTransformer(self.models_name[index])

    def __len__(self):
        return len(self.models_name)

models =[
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/sentence-t5-base",
    "intfloat/e5-base-v2",
    # "intfloat/multilingual-e5-base",
]

models_name = [model.split("/")[-1] for model in models]

model_iter = ModelIterator(models)

models_name

['all-MiniLM-L6-v2', 'sentence-t5-base', 'e5-base-v2']

In [16]:
import rdflib

def turtle_to_string(turtle_file_path: str) -> str:
    # Load the turtle file
    g = rdflib.Graph()
    g.parse(turtle_file_path, format="turtle")

    # Convert the turtle file to a string
    turtle_string = g.serialize(format="turtle")#.decode("utf-8")

    return turtle_string

pizza_onto_ground_truth_ttl = turtle_to_string("pizza_onto_ground_truth.ttl")

defect_onto_ground_truth_ttl = turtle_to_string("defect_onto_ground_truth.ttl")

complete_pipeline_kr_rdf_graph = turtle_to_string("complete_pipeline_kr_rdf_graph.ttl")

In [17]:

truth_ontologies  =  {
    "pizza_onto_ground_truth" :  pizza_onto_ground_truth,
    "defect_onto_ground_truth": defect_onto_ground_truth
}



multi_index = pd.MultiIndex.from_product([
   models_name,
   truth_ontologies.keys()
    ])

similarity_score = pd.DataFrame(index=["complete_pipeline_kr_rdf_graph"], columns=multi_index)


similarity_score

all-MiniLM-L6-v2  \
                               pizza_onto_ground_truth   
complete_pipeline_kr_rdf_graph                     NaN   

                                                         \
                               defect_onto_ground_truth   
complete_pipeline_kr_rdf_graph                      NaN   

                                      sentence-t5-base  \
                               pizza_onto_ground_truth   
complete_pipeline_kr_rdf_graph                     NaN   

                                                         \
                               defect_onto_ground_truth   
complete_pipeline_kr_rdf_graph                      NaN   

                                            e5-base-v2  \
                               pizza_onto_ground_truth   
complete_pipeline_kr_rdf_graph                     NaN   

                                                         
                               defect_onto_ground_truth  
complete_pipeline_kr_rdf_graph                      NaN

In [18]:
similarity_score.loc["complete_pipeline_kr_rdf_graph"]["all-MiniLM-L6-v2"] = [12, 25]
similarity_score

all-MiniLM-L6-v2  \
                               pizza_onto_ground_truth   
complete_pipeline_kr_rdf_graph                      12   

                                                         \
                               defect_onto_ground_truth   
complete_pipeline_kr_rdf_graph                       25   

                                      sentence-t5-base  \
                               pizza_onto_ground_truth   
complete_pipeline_kr_rdf_graph                     NaN   

                                                         \
                               defect_onto_ground_truth   
complete_pipeline_kr_rdf_graph                      NaN   

                                            e5-base-v2  \
                               pizza_onto_ground_truth   
complete_pipeline_kr_rdf_graph                     NaN   

                                                         
                               defect_onto_ground_truth  
complete_pipeline_kr_rdf_graph                      NaN

In [19]:


def compute_score(model, truth_ontology, ontology):
    # Encode the ontology

    # Compute the similarity between the ontology and the defect pipeline
    similarity = util.pytorch_cos_sim(truth_ontology, ontology)
    return float(similarity[0][0])

# scores = []

# for ontology_name, ontology in {"complete_pipeline_kr_rdf_graph": complete_pipeline_kr_rdf_graph}.items():
for idx, model in enumerate(model_iter):
    ontologies_embedding = {
        "pizza_onto_ground_truth": model.encode(pizza_onto_ground_truth),
        "defect_onto_ground_truth": model.encode(defect_onto_ground_truth),
        "complete_pipeline_kr_rdf_graph" : model.encode(complete_pipeline_kr_rdf_graph)
    }

    scores = []
    for truth_ontology in truth_ontologies.keys():
        score = compute_score(
            model,
            ontologies_embedding[truth_ontology],
            ontologies_embedding["complete_pipeline_kr_rdf_graph"]
            )
        scores.append(score)

    similarity_score.loc["complete_pipeline_kr_rdf_graph"][models_name[idx]] = scores


similarity_score

all-MiniLM-L6-v2  \
                               pizza_onto_ground_truth   
complete_pipeline_kr_rdf_graph                0.703732   

                                                         \
                               defect_onto_ground_truth   
complete_pipeline_kr_rdf_graph                 0.750401   

                                      sentence-t5-base  \
                               pizza_onto_ground_truth   
complete_pipeline_kr_rdf_graph                0.921615   

                                                         \
                               defect_onto_ground_truth   
complete_pipeline_kr_rdf_graph                 0.943632   

                                            e5-base-v2  \
                               pizza_onto_ground_truth   
complete_pipeline_kr_rdf_graph                0.882533   

                                                         
                               defect_onto_ground_truth  
complete_pipeline_kr_rdf_graph                 0.924053

In [20]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

In [21]:


def create_bar_chart(index_name, pipelines_scores):
    data = pipelines_scores.reset_index().melt(id_vars='index', var_name=["Modeles", "Ontologie"], value_name='Score')
    data.rename(columns={'index': 'built_ontology'}, inplace=True)

    df = data[data['built_ontology'] == index_name]
    fig = px.bar(df, x='Modeles', y='Score', color='Ontologie', barmode='group',
                 title=f"Scores de similarité entre l'ontologie obtenue et 2 ontologies de domaines differents")

    fig.update_layout(
        xaxis_title='Modeles',
        yaxis_title='Scores'
    )
    fig.update_layout(width=1000, height=600)
    fig.show()

create_bar_chart("complete_pipeline_kr_rdf_graph", similarity_score)


In [22]:
results = pd.DataFrame(
    index=["meta"],
    columns=["llm hierarchisation", "subsumtion hierarchisation"]
    )

multi_index = pd.MultiIndex.from_product([
    ["llm hierarchisation", "subsumtion hierarchisation"],
    ["Precision", "Rappel", "F1"]
])

meta_scores = pd.DataFrame(index=["meta"], columns=multi_index)

meta_scores.loc["meta"] = [0.2391304347826087, 0.5, 0.3235294117647059, 0.02857142857142857, 0.045454545454545456, 0.03508771929824561]

In [23]:

def create_bar_chart_meta(index_name, pipelines_scores):
    data = pipelines_scores.reset_index().melt(id_vars='index', var_name=["Composants", "Metrique"], value_name='Score')
    data.rename(columns={'index': 'built_ontology'}, inplace=True)

    df = data[data['built_ontology'] == index_name]
    fig = px.bar(df, x='Composants', y='Score', color='Metrique', barmode='group',
                 title=f"Scores de Précision, Rappel et F1 pour l'extraction de metarelations")

    fig.update_layout(
        xaxis_title='Composants',
        yaxis_title='Scores'
    )
    fig.update_layout(width=1000, height=600)
    fig.show()

create_bar_chart_meta("meta", meta_scores)